In [1]:
# ============================================
# LEGAL-BERT IMPLEMENTATION
# Week 5-8: Advanced Feature Extraction
# ============================================

print("=" * 70)
print("LEGAL-BERT FEATURE EXTRACTION")
print("=" * 70)
print()

# Install required packages (run once)
# Uncomment these lines if not installed:
# !pip install transformers
# !pip install sentence-transformers
# !pip install torch
# !pip install pandas numpy scikit-learn

# Import libraries
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import pickle
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print()


LEGAL-BERT FEATURE EXTRACTION

✅ Libraries imported successfully
PyTorch version: 2.9.0+cpu
CUDA available: False



In [2]:
# ============================================
# LOAD CLEANED DATASET & PREVIOUS FEATURES
# ============================================

print("STEP 1: Loading cleaned dataset...")
print("-" * 70)

# Load original cleaned data (for text columns)
df = pd.read_csv('dataset_cleaned_v2.csv')

# Load previously created features (from feature engineering)
X_train_prev = pd.read_csv('X_train.csv')
X_test_prev = pd.read_csv('X_test.csv')

# Load targets
y_train_outcome = np.load('y_train_outcome.npy')
y_test_outcome = np.load('y_test_outcome.npy')

print(f"✅ Loaded dataset: {len(df)} total records")
print(f"✅ Training set: {len(X_train_prev)} samples")
print(f"✅ Test set: {len(X_test_prev)} samples")
print(f"✅ Previous features: {len(X_train_prev.columns)}")
print()

# Identify text columns
text_columns = [
    'brief_facts_summary',
    'grounds_of_appeal_raw_text_summary', 
    'court_of_appeal_analysis_summary'
]

# Create combined text for each case
df['combined_text'] = (
    df['brief_facts_summary'].fillna('') + ' ' +
    df['grounds_of_appeal_raw_text_summary'].fillna('') + ' ' +
    df['court_of_appeal_analysis_summary'].fillna('')
)

print(f"✅ Text columns prepared")
print(f"   Average text length: {df['combined_text'].str.len().mean():.0f} characters")
print()


STEP 1: Loading cleaned dataset...
----------------------------------------------------------------------
✅ Loaded dataset: 1251 total records
✅ Training set: 1000 samples
✅ Test set: 251 samples
✅ Previous features: 59

✅ Text columns prepared
   Average text length: 1505 characters



In [3]:
# ============================================
# LOAD LEGAL-BERT MODEL
# ============================================

print("STEP 2: Loading Legal-BERT model...")
print("-" * 70)

# Model name (trained on legal documents!)
model_name = "nlpaueb/legal-bert-base-uncased"

print(f"Loading: {model_name}")
print("This may take 2-3 minutes on first run (downloading 440MB)...")
print()

try:
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    
    # Move to GPU if available
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()  # Set to evaluation mode
    
    print(f"✅ Legal-BERT loaded successfully!")
    print(f"   Device: {device}")
    print(f"   Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print()
    
except Exception as e:
    print(f"❌ Error loading Legal-BERT: {e}")
    print("Trying alternative approach...")
    
    # Fallback: Use Sentence-BERT (easier to use)
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    model = SentenceTransformer(model_name)
    print(f"✅ Using Sentence-BERT instead: {model_name}")
    print()


STEP 2: Loading Legal-BERT model...
----------------------------------------------------------------------
Loading: nlpaueb/legal-bert-base-uncased
This may take 2-3 minutes on first run (downloading 440MB)...



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

✅ Legal-BERT loaded successfully!
   Device: cpu
   Model parameters: 109,482,240



Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Error while downloading from https://huggingface.co/nlpaueb/legal-bert-base-uncased/resolve/refs%2Fpr%2F1/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


In [4]:
# ============================================
# GENERATE LEGAL-BERT EMBEDDINGS
# ============================================

print("STEP 3: Generating BERT embeddings...")
print("-" * 70)

def get_bert_embedding(text, max_length=512):
    """
    Generate BERT embedding for a single text
    
    Parameters:
    - text: Input text string
    - max_length: Maximum tokens (Legal-BERT limit = 512)
    
    Returns:
    - embedding: 768-dimensional vector
    """
    # Tokenize
    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
        padding='max_length'
    )
    
    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Generate embedding
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Use [CLS] token embedding (first token)
    # This represents the entire sentence
    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    
    return embedding.flatten()

# Test on one example
sample_text = df['combined_text'].iloc[0][:500]  # First 500 chars
sample_embedding = get_bert_embedding(sample_text)

print(f"✅ Embedding function works!")
print(f"   Input length: {len(sample_text)} characters")
print(f"   Output shape: {sample_embedding.shape}")
print(f"   Sample values: {sample_embedding[:5]}")
print()


STEP 3: Generating BERT embeddings...
----------------------------------------------------------------------
✅ Embedding function works!
   Input length: 500 characters
   Output shape: (768,)
   Sample values: [ 0.4746084  -0.4073741  -0.06696382 -0.35877678 -0.14459847]



In [5]:
# ============================================
# GENERATE EMBEDDINGS FOR ALL CASES
# ============================================

print("STEP 4: Processing all cases...")
print("-" * 70)

# Sort dataframe same way as train/test split
df_sorted = df.sort_values('coa_year').reset_index(drop=True)

# Split indices (same 80/20 as before)
split_idx = int(len(df_sorted) * 0.8)

print(f"Total cases to process: {len(df_sorted)}")
print(f"Training cases: {split_idx}")
print(f"Test cases: {len(df_sorted) - split_idx}")
print()

# Generate embeddings (this will take time!)
print("Generating embeddings (estimated time: 10-15 minutes)...")
print("Progress:")

embeddings_list = []
batch_size = 10  # Process 10 at a time

for i in range(0, len(df_sorted), batch_size):
    batch_texts = df_sorted['combined_text'].iloc[i:i+batch_size]
    
    for text in batch_texts:
        # Truncate long texts
        text_truncated = text[:2000]  # First 2000 chars
        embedding = get_bert_embedding(text_truncated)
        embeddings_list.append(embedding)
    
    # Progress update
    if (i + batch_size) % 100 == 0:
        print(f"   Processed {i + batch_size}/{len(df_sorted)} cases...")

print()
print(f"✅ All embeddings generated!")

# Convert to numpy array
bert_embeddings = np.array(embeddings_list)
print(f"   Shape: {bert_embeddings.shape}")  # Should be (1251, 768)
print()


STEP 4: Processing all cases...
----------------------------------------------------------------------
Total cases to process: 1251
Training cases: 1000
Test cases: 251

Generating embeddings (estimated time: 10-15 minutes)...
Progress:


model.safetensors:  64%|######4   | 283M/440M [00:00<?, ?B/s]

   Processed 100/1251 cases...
   Processed 200/1251 cases...
   Processed 300/1251 cases...
   Processed 400/1251 cases...
   Processed 500/1251 cases...
   Processed 600/1251 cases...
   Processed 700/1251 cases...
   Processed 800/1251 cases...
   Processed 900/1251 cases...
   Processed 1000/1251 cases...
   Processed 1100/1251 cases...
   Processed 1200/1251 cases...

✅ All embeddings generated!
   Shape: (1251, 768)



In [6]:
# ============================================
# SPLIT BERT EMBEDDINGS INTO TRAIN/TEST
# ============================================

print("STEP 5: Splitting embeddings...")
print("-" * 70)

# Split embeddings (same 80/20 split)
bert_train = bert_embeddings[:split_idx]
bert_test = bert_embeddings[split_idx:]

print(f" Training embeddings: {bert_train.shape}")
print(f" Test embeddings: {bert_test.shape}")
print()

# Convert to DataFrame with column names
bert_train_df = pd.DataFrame(
    bert_train,
    columns=[f'bert_{i}' for i in range(bert_train.shape[1])]
)

bert_test_df = pd.DataFrame(
    bert_test,
    columns=[f'bert_{i}' for i in range(bert_test.shape[1])]
)

print(f" Converted to DataFrames")
print(f"   Training: {bert_train_df.shape}")
print(f"   Test: {bert_test_df.shape}")
print()


STEP 5: Splitting embeddings...
----------------------------------------------------------------------
 Training embeddings: (1000, 768)
 Test embeddings: (251, 768)

 Converted to DataFrames
   Training: (1000, 768)
   Test: (251, 768)



In [7]:
# ============================================
# COMBINE BERT + PREVIOUS FEATURES
# ============================================

print("STEP 6: Combining BERT with previous features...")
print("-" * 70)

# Reset indices
X_train_prev = X_train_prev.reset_index(drop=True)
X_test_prev = X_test_prev.reset_index(drop=True)
bert_train_df = bert_train_df.reset_index(drop=True)
bert_test_df = bert_test_df.reset_index(drop=True)

# Combine
X_train_final = pd.concat([X_train_prev, bert_train_df], axis=1)
X_test_final = pd.concat([X_test_prev, bert_test_df], axis=1)

print(f"✅ Combined features:")
print(f"   Training: {X_train_final.shape}")
print(f"   Test: {X_test_final.shape}")
print()

print("Feature breakdown:")
print(f"   - Previous features: {len(X_train_prev.columns)}")
print(f"   - BERT features: {len(bert_train_df.columns)}")
print(f"   - TOTAL: {len(X_train_final.columns)}")
print()


STEP 6: Combining BERT with previous features...
----------------------------------------------------------------------
✅ Combined features:
   Training: (1000, 827)
   Test: (251, 827)

Feature breakdown:
   - Previous features: 59
   - BERT features: 768
   - TOTAL: 827



In [8]:
# ============================================
# SAVE BERT-ENHANCED FEATURES
# ============================================

print("STEP 7: Saving BERT-enhanced features...")
print("-" * 70)

# Save combined features
X_train_final.to_csv('X_train_bert.csv', index=False)
X_test_final.to_csv('X_test_bert.csv', index=False)

# Save raw BERT embeddings (for similarity search later)
np.save('bert_embeddings_train.npy', bert_train)
np.save('bert_embeddings_test.npy', bert_test)
np.save('bert_embeddings_all.npy', bert_embeddings)

# Save model info
bert_metadata = {
    'model_name': model_name,
    'embedding_dim': 768,
    'total_features': int(len(X_train_final.columns)),
    'bert_features': 768,
    'previous_features': int(len(X_train_prev.columns)),
    'train_samples': int(len(X_train_final)),
    'test_samples': int(len(X_test_final)),
    'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
}

import json
with open('bert_metadata.json', 'w') as f:
    json.dump(bert_metadata, f, indent=4)

print("✅ Saved files:")
print("   - X_train_bert.csv (1000 samples, 827 features)")
print("   - X_test_bert.csv (251 samples, 827 features)")
print("   - bert_embeddings_*.npy (raw embeddings)")
print("   - bert_metadata.json (documentation)")
print()

print("=" * 70)
print("🎉 LEGAL-BERT FEATURE EXTRACTION COMPLETE!")
print("=" * 70)
print(f"✅ Total features: {len(X_train_final.columns)}")
print(f"✅ BERT embeddings: 768 dimensions")
print(f"✅ Ready for modeling!")
print("=" * 70)


STEP 7: Saving BERT-enhanced features...
----------------------------------------------------------------------
✅ Saved files:
   - X_train_bert.csv (1000 samples, 827 features)
   - X_test_bert.csv (251 samples, 827 features)
   - bert_embeddings_*.npy (raw embeddings)
   - bert_metadata.json (documentation)

🎉 LEGAL-BERT FEATURE EXTRACTION COMPLETE!
✅ Total features: 827
✅ BERT embeddings: 768 dimensions
✅ Ready for modeling!
